In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS rahul_de_databricks.customer_data")

DataFrame[]

In [0]:
spark.sql("SHOW SCHEMAS IN rahul_de_databricks").show()

+------------------+
|      databaseName|
+------------------+
|     customer_data|
|           default|
|information_schema|
+------------------+



In [0]:
df_customer = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://landing@rahulde2026lake.dfs.core.windows.net/processed/customer1.csv")

df_customer.show()

+----------+---------+--------+--------------------+---------+-------------------+------------+
|CustomerID|FirstName|LastName|               Email|     City|         SignupDate|    FullName|
+----------+---------+--------+--------------------+---------+-------------------+------------+
|         1|    Aarav|  Sharma|aarav.sharma@emai...|   Mumbai|2024-01-15 00:00:00|Aarav Sharma|
|         2|    Priya|    Nair|priya.nair@email.com|Bengaluru|2024-02-20 00:00:00|  Priya Nair|
|         3|    Rohan|   Mehta|rohan.mehta@email...|    Delhi|2024-03-05 00:00:00| Rohan Mehta|
|         4|   Ananya|    Iyer|ananya.iyer@email...|  Chennai|2024-03-22 00:00:00| Ananya Iyer|
|         5|   Vikram|   Singh|vikram.singh@emai...|     Pune|2024-04-10 00:00:00|Vikram Singh|
|         6|    Sneha|   Reddy|sneha.reddy@email...|Hyderabad|2024-05-01 00:00:00| Sneha Reddy|
|         7|    Karan|   Joshi|karan.joshi@email...|Ahmedabad|2024-05-18 00:00:00| Karan Joshi|
|         8|     Diya|  Kapoor|diya.kapo

In [0]:
df_customer.write.mode("overwrite").saveAsTable("rahul_de_databricks.customer_data.customers_bronze")

In [0]:
spark.sql("SELECT * FROM rahul_de_databricks.customer_data.customers_bronze").show()

+----------+---------+--------+--------------------+---------+-------------------+------------+
|CustomerID|FirstName|LastName|               Email|     City|         SignupDate|    FullName|
+----------+---------+--------+--------------------+---------+-------------------+------------+
|         1|    Aarav|  Sharma|aarav.sharma@emai...|   Mumbai|2024-01-15 00:00:00|Aarav Sharma|
|         2|    Priya|    Nair|priya.nair@email.com|Bengaluru|2024-02-20 00:00:00|  Priya Nair|
|         3|    Rohan|   Mehta|rohan.mehta@email...|    Delhi|2024-03-05 00:00:00| Rohan Mehta|
|         4|   Ananya|    Iyer|ananya.iyer@email...|  Chennai|2024-03-22 00:00:00| Ananya Iyer|
|         5|   Vikram|   Singh|vikram.singh@emai...|     Pune|2024-04-10 00:00:00|Vikram Singh|
|         6|    Sneha|   Reddy|sneha.reddy@email...|Hyderabad|2024-05-01 00:00:00| Sneha Reddy|
|         7|    Karan|   Joshi|karan.joshi@email...|Ahmedabad|2024-05-18 00:00:00| Karan Joshi|
|         8|     Diya|  Kapoor|diya.kapo

In [0]:
from pyspark.sql.functions import when, col

df_customer2 = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://landing@rahulde2026lake.dfs.core.windows.net/processed/customer1.csv")

# Add one extra column — a simple derived flag based on City
df_customer2 = df_customer2.withColumn(
    "IsMetro",
    when(col("City").isin("Mumbai", "Delhi", "Bengaluru", "Chennai", "Hyderabad", "Kolkata"), "Yes")
    .otherwise("No")
)

df_customer2.show()

+----------+---------+--------+--------------------+---------+-------------------+------------+-------+
|CustomerID|FirstName|LastName|               Email|     City|         SignupDate|    FullName|IsMetro|
+----------+---------+--------+--------------------+---------+-------------------+------------+-------+
|         1|    Aarav|  Sharma|aarav.sharma@emai...|   Mumbai|2024-01-15 00:00:00|Aarav Sharma|    Yes|
|         2|    Priya|    Nair|priya.nair@email.com|Bengaluru|2024-02-20 00:00:00|  Priya Nair|    Yes|
|         3|    Rohan|   Mehta|rohan.mehta@email...|    Delhi|2024-03-05 00:00:00| Rohan Mehta|    Yes|
|         4|   Ananya|    Iyer|ananya.iyer@email...|  Chennai|2024-03-22 00:00:00| Ananya Iyer|    Yes|
|         5|   Vikram|   Singh|vikram.singh@emai...|     Pune|2024-04-10 00:00:00|Vikram Singh|     No|
|         6|    Sneha|   Reddy|sneha.reddy@email...|Hyderabad|2024-05-01 00:00:00| Sneha Reddy|    Yes|
|         7|    Karan|   Joshi|karan.joshi@email...|Ahmedabad|20

In [0]:
df_customer2.write.mode("overwrite").saveAsTable("rahul_de_databricks.customer_data.customers_enriched")

In [0]:
spark.sql("SELECT * FROM rahul_de_databricks.customer_data.customers_enriched").show()

+----------+---------+--------+--------------------+---------+-------------------+------------+-------+
|CustomerID|FirstName|LastName|               Email|     City|         SignupDate|    FullName|IsMetro|
+----------+---------+--------+--------------------+---------+-------------------+------------+-------+
|         1|    Aarav|  Sharma|aarav.sharma@emai...|   Mumbai|2024-01-15 00:00:00|Aarav Sharma|    Yes|
|         2|    Priya|    Nair|priya.nair@email.com|Bengaluru|2024-02-20 00:00:00|  Priya Nair|    Yes|
|         3|    Rohan|   Mehta|rohan.mehta@email...|    Delhi|2024-03-05 00:00:00| Rohan Mehta|    Yes|
|         4|   Ananya|    Iyer|ananya.iyer@email...|  Chennai|2024-03-22 00:00:00| Ananya Iyer|    Yes|
|         5|   Vikram|   Singh|vikram.singh@emai...|     Pune|2024-04-10 00:00:00|Vikram Singh|     No|
|         6|    Sneha|   Reddy|sneha.reddy@email...|Hyderabad|2024-05-01 00:00:00| Sneha Reddy|    Yes|
|         7|    Karan|   Joshi|karan.joshi@email...|Ahmedabad|20

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW rahul_de_databricks.customer_data.metro_customers_view AS
SELECT CustomerID, FirstName, LastName, City, IsMetro
FROM rahul_de_databricks.customer_data.customers_enriched
WHERE IsMetro = 'Yes'
""")

DataFrame[]

In [0]:
spark.sql("SELECT * FROM rahul_de_databricks.customer_data.metro_customers_view").show()

+----------+---------+--------+---------+-------+
|CustomerID|FirstName|LastName|     City|IsMetro|
+----------+---------+--------+---------+-------+
|         1|    Aarav|  Sharma|   Mumbai|    Yes|
|         2|    Priya|    Nair|Bengaluru|    Yes|
|         3|    Rohan|   Mehta|    Delhi|    Yes|
|         4|   Ananya|    Iyer|  Chennai|    Yes|
|         6|    Sneha|   Reddy|Hyderabad|    Yes|
|         8|     Diya|  Kapoor|  Kolkata|    Yes|
|         9|    Arjun|  Pillai|   Mumbai|    Yes|
|        10|    Meera|     Rao|Bengaluru|    Yes|
+----------+---------+--------+---------+-------+



In [0]:
spark.sql("""
CREATE OR REPLACE FUNCTION rahul_de_databricks.customer_data.classify_signup(signup_date DATE)
RETURNS STRING
RETURN CASE
    WHEN signup_date >= '2024-06-01' THEN 'Recent'
    ELSE 'Older'
END
""")

DataFrame[]

In [0]:
spark.sql("""
SELECT CustomerID, FirstName, SignupDate,
       rahul_de_databricks.customer_data.classify_signup(SignupDate) AS SignupCategory
FROM rahul_de_databricks.customer_data.customers_bronze
""").show()

+----------+---------+-------------------+--------------+
|CustomerID|FirstName|         SignupDate|SignupCategory|
+----------+---------+-------------------+--------------+
|         1|    Aarav|2024-01-15 00:00:00|         Older|
|         2|    Priya|2024-02-20 00:00:00|         Older|
|         3|    Rohan|2024-03-05 00:00:00|         Older|
|         4|   Ananya|2024-03-22 00:00:00|         Older|
|         5|   Vikram|2024-04-10 00:00:00|         Older|
|         6|    Sneha|2024-05-01 00:00:00|         Older|
|         7|    Karan|2024-05-18 00:00:00|         Older|
|         8|     Diya|2024-06-02 00:00:00|        Recent|
|         9|    Arjun|2024-06-25 00:00:00|        Recent|
|        10|    Meera|2024-07-08 00:00:00|        Recent|
+----------+---------+-------------------+--------------+

